# 1 — Preparing training data

Turns a corpus of recordings and transcripts into the lists StyleTTS2 trains on.

**No data ships with this repository.** See `DATA.md` for how to obtain a corpus; the
UltraSuite benchmark on the Hugging Face Hub is a reasonable starting point. Set
`data_root` in your `paths.yaml` before running this.

In [ ]:
from chiressd.paths import load_paths

paths = load_paths()
paths.source, paths.data_root

## Input format

One row per utterance, pipe-delimited:

```
filename.wav|the spoken text|speaker_id
```

`speaker_id` is optional and defaults to 0.

In [ ]:
from chiressd.data import read_metadata

metadata_path = paths.data_root / "clp" / "metadata.txt"
rows = read_metadata(metadata_path)
print(f"{len(rows)} rows")
rows[:3]

## Phonemize

The middle field of a training list is **phonemes, not text**. Getting this wrong fails
quietly: training proceeds and simply learns from mis-specified input.

Two defaults here are worth knowing about:

- `with_stress=False` — training lists carry no stress marks, although synthesis
  phonemizes *with* them. That asymmetry is inherited from the original pipeline.
- `language="en-gb"` — the target speakers have a Central Scottish accent, which British
  English approximates considerably better than American English. Change it to match your
  own speakers.

Pass `with_stress=True, tokenize=True` if you would rather the two paths agree.

In [ ]:
from chiressd.data import build_entries

entries = build_entries(
    rows,
    audio_dir=paths["datasets.train.audio"],
    language="en-gb",
    progress=True,
)
print(f"{len(entries)} usable utterances")
print(entries[0].to_line())

## Split and write

The shuffle is seeded so the split is reproducible. Splitting *without* shuffling would
follow filename order, which in these corpora groups by speaker and session — so the
validation set would be a handful of speakers rather than a sample of the corpus.

In [ ]:
from chiressd.data import split_entries, write_list

train, val = split_entries(entries, val_fraction=0.15, seed=0)

train_path = write_list(paths["datasets.train.train_list"], train)
val_path = write_list(paths["datasets.train.val_list"], val)
print(f"train: {len(train)} -> {train_path}")
print(f"val:   {len(val)} -> {val_path}")

The same thing from the command line:

```bash
chiressd-prepare metadata.txt
```

## Next

Fine-tune with the released recipe:

```bash
chiressd-train --config config_ft_F0_v5 --dry-run   # inspect first
chiressd-train --config config_ft_F0_v5
```

`--dry-run` renders the config and stops. Every real run writes its rendered config
beside the output, which is the record of what the run actually used.